# E-commerce Marketing Analytics | January–June 2026

**Portfolio case study:** Python-based source validation, channel attribution, purchase-funnel analysis, customer behaviour, revenue sensitivity and decision-focused Q3 budget scenarios.

**Source:** `Shop_online_data.xlsx` (synthetic data, not redistributed). Required sheets: `orders`, `customers`, `marketing_spend`, `sessions`, `Data dictionary`. **Management report:** `marketing_management_report.xlsx`.

**Financial policy:** Count unique positive completed orders as purchases; separately deduct *all 12 negative adjustments (−€719.51)*, including negative returned/cancelled records, from attributed revenue. Positive cancelled/returned orders are excluded. Both conflicting `O500891` records are quarantined. Keep the valid €24,990 Meta purchase in primary revenue and remove it only for diagnostic sensitivity.

**Decision policy:** Compare a constant €52,853.72 Q2-equivalent envelope with three possible €624.09 transfers out of Meta: 60/40 Affiliate/Google, Affiliate-only, and Google-only. **Affiliate-only is preferred conditionally; HOLD UPLIFT until Finance/Marketing approval.** No causal or incremental-profit claims.

**Output status:** The source workbook is not available in this portfolio package. Consequently, the updated source-level code has **not been rerun** here, and stale outputs from the earlier treatment have been cleared. The benchmark figures below are taken directly from the final Excel management report, **not asserted to be newly executed notebook results**. Run cells top-to-bottom with the permitted source workbook to verify them.

## Final Excel report — independently inspectable reference figures

| Measure (January–June 2026) | Final report |
|---|---:|
| Positive completed orders | 3,065 |
| Positive completed sales | €212,282.52 |
| All recorded negative adjustments | −€719.51 (12 records) |
| **Net attributed revenue** | **€211,563.01** |
| Adjusted paid-channel revenue | €168,265.85 |
| Recorded paid-media spend | €103,861.75 |
| **Paid-channel attributed ROAS** | **1.620×** |
| Meta ROAS, including €24,990 sale | 1.372× |
| Meta ROAS, diagnostic exclusion | 0.825× |
| Q2 paid budget | €52,853.72 |
| Proposed transfer from Meta | €624.09 |
| Approval status | **HOLD UPLIFT** |

Q2-equivalent alternatives: A (60/40) = **€374.45 Affiliate + €249.64 Google**; B (preferred, conditional) = **€624.09 Affiliate**; C = **€624.09 Google**. Every alternative keeps total paid spend unchanged and is an allocation illustration, not a profit forecast.

![Adjusted-revenue Meta sensitivity](meta_roas_sensitivity_audited.png)

![Session-to-cart rates](session_to_cart_by_channel.png)

The code below independently recomputes these figures **when the original permitted source data is supplied**. Failure to reproduce any benchmark is treated as a discrepancy to investigate, not silently rounded away.

## 1. Source data and reporting scope

In [ ]:
# Load source tables at their native transaction/session grains; retain row references for exception tracing.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

source_file = Path("Shop_online_data.xlsx")
assert source_file.exists(), "Place Shop_online_data.xlsx beside this notebook."

orders_raw = pd.read_excel(source_file, sheet_name="orders")
spend_raw = pd.read_excel(source_file, sheet_name="marketing_spend")
sessions = pd.read_excel(source_file, sheet_name="sessions")
customers = pd.read_excel(source_file, sheet_name="customers")
data_dictionary = pd.read_excel(source_file, sheet_name="Data dictionary")

orders_raw["Excel row"] = orders_raw.index + 2  # Excel row 1 is the header.
spend_raw["Excel row"] = spend_raw.index + 2

print("Source row counts: orders =", len(orders_raw),
      "| spend =", len(spend_raw),
      "| sessions =", len(sessions),
      "| customers =", len(customers))
display(orders_raw.head(5))


## 2. Source data quality and revenue distribution

Inspect field completeness, statuses and transaction extremes **before** applying financial rules. Positive sales and negative adjustments have different eligibility conditions: positive cancelled/returned orders are excluded; negative entries of any status are deducted in adjusted attributed revenue, subject to source-ledger confirmation.

In [ ]:
# Profile input completeness, statuses and source exceptions before eligibility changes.
missing_values = pd.DataFrame({
    "Orders": orders_raw.isna().sum(), "Spend": spend_raw.isna().sum(),
    "Sessions": sessions.isna().sum(), "Customers": customers.isna().sum()
}).fillna(0).astype(int)
print("Missing source values (nonzero counts):")
display(missing_values[missing_values.sum(axis=1) > 0])
print("Raw order status counts:")
display(orders_raw["status"].value_counts().to_frame("Rows"))
print("Raw revenue percentiles (includes credits, duplicates and noncompleted orders):")
display(orders_raw["revenue"].describe(percentiles=[.25,.5,.75,.9,.95,.99]).round(2))
print("Source orders above €300:")
display(orders_raw.loc[orders_raw["revenue"].gt(300),
    ["Excel row", "order_id", "customer_id", "order_date", "channel", "revenue", "status"]]
    .sort_values("revenue", ascending=False))
raw_negative = orders_raw.loc[orders_raw["revenue"].lt(0)].copy()
print("All negative source rows by status (before deduplicating):")
display(raw_negative.groupby("status")["revenue"].agg(Records="size", Amount_EUR="sum"))
print("The negative adjustment policy is applied only AFTER conflicting IDs and duplicates are handled.")

## 3. Duplicate records and conflicting transactions

In [ ]:
# Identify duplicate transaction identifiers and revenue conflicts independently of the deduplication rule.
# Matching spend records are only presumed duplicate exports because no spend transaction ID is provided.
repeat_orders = orders_raw.loc[
    orders_raw.duplicated(subset="order_id", keep=False)
].sort_values(["order_id", "Excel row"]).copy()

conflict_counts = repeat_orders.groupby("order_id")["revenue"].nunique()
conflicting_ids = conflict_counts[conflict_counts > 1].index.tolist()

spend_fields = ["date", "channel", "campaign", "spend", "clicks", "impressions"]
repeat_spend = spend_raw.loc[
    spend_raw.duplicated(subset=spend_fields, keep=False)
].copy()

print("Repeated order IDs:", repeat_orders["order_id"].nunique())
print("Order rows involved:", len(repeat_orders))
print("Conflicting revenue IDs:", conflicting_ids)
print("Identical spend rows involved:", len(repeat_spend))

display(repeat_orders.loc[repeat_orders["order_id"].isin(conflicting_ids)])

assert set(conflicting_ids) == {"O500891"}, "Unexpected conflicting IDs: review the input."


## 4. Order and expenditure deduplication

`O500891` has contradictory values for the same order ID, so both source records are quarantined pending transaction-level verification. Other identical order copies are counted once. Exact matching spend rows are provisionally deduplicated; absent a spend-ledger identifier, this remains an explicit assumption.

In [ ]:
# Quarantine both contradictory O500891 rows; deduplicate only nonconflicting orders.
# Provisionally collapse exact spend-row copies while retaining them for review.
conflicting_order = orders_raw.loc[orders_raw["order_id"] == "O500891"].copy()

orders_no_conflict = orders_raw.loc[
    orders_raw["order_id"] != "O500891"
].copy()

extra_order_copies = orders_no_conflict.loc[
    orders_no_conflict.duplicated(subset="order_id", keep="first")
].copy()
orders_clean = orders_no_conflict.drop_duplicates(
    subset="order_id", keep="first"
).copy()

extra_spend_copies = spend_raw.loc[
    spend_raw.duplicated(subset=spend_fields, keep="first")
].copy()
spend_clean = spend_raw.drop_duplicates(subset=spend_fields).copy()

print("O500891 rows excluded:", len(conflicting_order))
print("Other repeated order copies removed:", len(extra_order_copies))
print("Identical spend copies removed:", len(extra_spend_copies))
print("Remaining unique order IDs:", len(orders_clean))

assert len(conflicting_order) == 2
assert "O500891" not in orders_clean["order_id"].values
assert orders_clean["order_id"].is_unique


## 5. Channel mapping and time fields

In [ ]:
# Map variant channel labels to one reporting taxonomy before grouping across tables.
channel_names = {
    "fb": "Meta", "facebook": "Meta", "meta": "Meta",
    "google ads": "Google Ads", "google": "Google Ads", "googleads": "Google Ads",
    "e-mail": "Email", "email": "Email", "newsletter": "Email",
    "affiliate": "Affiliate", "afiliate": "Affiliate", "organic": "Organic"
}

for table, column in [
    (orders_clean, "channel"), (spend_clean, "channel"),
    (sessions, "channel"), (customers, "acquisition_channel")
]:
    table["channel_clean"] = table[column].astype(str).str.strip().str.lower().map(channel_names)
    assert table["channel_clean"].notna().all(), "Unknown channel name: check spelling."

orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])
spend_clean["date"] = pd.to_datetime(spend_clean["date"])
sessions["date"] = pd.to_datetime(sessions["date"])
customers["registration_date"] = pd.to_datetime(customers["registration_date"])

channels = ["Email", "Affiliate", "Google Ads", "Meta", "Organic"]
print("Normalised channels:", channels)


## 6. Completed-order EDA

Order-value summaries use unique, positive, completed orders with the disputed ID excluded. Large completed purchases remain valid for the primary scenario unless contrary source evidence is available. The IQR threshold is diagnostic only—not a rule for rejecting revenue.

In [ ]:
# Report completed-order distribution and IQR diagnostics without automatically rejecting high-value sales.
# The histogram truncates the displayed range, not the underlying data.
positive_completed = orders_clean.loc[
    orders_clean["status"].eq("completed") & orders_clean["revenue"].gt(0)
].copy()

eda_revenue = positive_completed["revenue"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)
display(eda_revenue.to_frame("Completed positive revenue (€)"))

q1 = positive_completed["revenue"].quantile(0.25)
q3 = positive_completed["revenue"].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"Median order: €{positive_completed['revenue'].median():,.2f}")
print(f"IQR statistical upper fence: €{upper_fence:,.2f}")
print("Orders above the IQR fence:", (positive_completed["revenue"] > upper_fence).sum())
print("Largest completed orders:")
display(positive_completed.nlargest(10, "revenue")[[
    "order_id", "channel_clean", "order_date", "revenue", "status"
]])

ordinary_sizes = positive_completed.loc[
    positive_completed["revenue"] <= 300, "revenue"
]
plt.figure(figsize=(8, 4))
plt.hist(ordinary_sizes, bins=25, edgecolor="white")
plt.title("Completed order values up to €300 (for a readable chart)")
plt.xlabel("Order revenue (€)")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()


## 7. Revenue and order eligibility policy

**Purchases:** positive, completed, unique orders only (including `O502653`, €24,990 Meta). **Adjusted attributed revenue:** the same positive purchases plus all retained negative-value adjustments, regardless of status. This prevents incorrectly dropping returned/cancelled credits while keeping positive cancelled/returned orders out of sales. Negative adjustments are not additional purchases; they affect revenue, ROAS and time series only. `O500891` remains quarantined.

In [ ]:
# Separate purchased-order population from signed financial activity.
meta_id = "O502653"
valid_orders = positive_completed.copy()   # Used for counts, purchase mix and customer cohorts.
negative_adjustments = orders_clean.loc[orders_clean["revenue"].lt(0)].copy()
revenue_records = pd.concat([valid_orders, negative_adjustments], ignore_index=True)

assert "O500891" not in revenue_records["order_id"].values
assert meta_id in valid_orders["order_id"].values
assert float(valid_orders.loc[valid_orders["order_id"] == meta_id, "revenue"].iloc[0]) == 24990.0
print("Positive completed purchases:",len(valid_orders),
      "| positive sales €{:,.2f}".format(valid_orders["revenue"].sum()))
print("Recorded negative adjustments:",len(negative_adjustments),
      "| signed value €{:,.2f}".format(negative_adjustments["revenue"].sum()))
print("Adjusted attributed revenue: €{:,.2f}".format(revenue_records["revenue"].sum()))

# Bookkeeping checks based on the published final report. These validate the reproduction,
# not the independent truth of disputed source records.
assert len(valid_orders) == 3065
assert len(negative_adjustments) == 12
assert abs(valid_orders["revenue"].sum()-212282.52)<0.01
assert abs(negative_adjustments["revenue"].sum()+719.51)<0.01
assert abs(revenue_records["revenue"].sum()-211563.01)<0.01

## 8. Question 1 — channel efficiency and order quality

Channel-attributed revenue is compared with separately recorded advertising spend. ROAS reflects historical attribution, not incremental profitability. A secondary order-mix and status analysis distinguishes repeat purchasing activity from the share of unique orders cancelled or returned; neither is a causal measure of channel acquisition quality.

In [ ]:
# Reconcile signed attributed revenue (positive sales plus negative adjustments) with separately aggregated spend.
# Organic paid-media ratios are undefined because no paid spend is recorded.
channel_report = pd.DataFrame(index=channels)
channel_report.index.name = "Channel"
channel_report["Revenue (€)"] = revenue_records.groupby("channel_clean")["revenue"].sum()
channel_report["Completed orders"] = valid_orders.groupby("channel_clean")["order_id"].count()
spend_by_channel = spend_clean.groupby("channel_clean")[["spend", "clicks", "impressions"]].sum()
channel_report["Spend (€)"] = spend_by_channel["spend"].reindex(channels).fillna(0)
channel_report["Clicks"] = spend_by_channel["clicks"].reindex(channels).fillna(0)
channel_report["Impressions"] = spend_by_channel["impressions"].reindex(channels).fillna(0)
channel_report[["Revenue (€)", "Completed orders"]] = channel_report[["Revenue (€)", "Completed orders"]].fillna(0)

paid_cost = channel_report["Spend (€)"].replace(0, float("nan"))
channel_report["ROAS"] = channel_report["Revenue (€)"] / paid_cost
channel_report["Spend per order (€)"] = paid_cost / channel_report["Completed orders"].replace(0, float("nan"))
channel_report["CTR"] = channel_report["Clicks"] / channel_report["Impressions"].replace(0, float("nan"))
channel_report["CPC (€)"] = channel_report["Spend (€)"] / channel_report["Clicks"].replace(0, float("nan"))

print("Adjusted six-month channel comparison (NaN = not measurable; Organic spend not recorded):")
display(channel_report[["Spend (€)", "Revenue (€)", "Completed orders", "ROAS",
                        "Spend per order (€)", "CTR", "CPC (€)"]].round(3))
channel_report["ROAS"].dropna().sort_values().plot(kind="barh", figsize=(8, 3),
    title="Attributed revenue / advertising spend, by paid channel")
plt.xlabel("ROAS (×)")
plt.tight_layout()
plt.show()


# Distinguish first-observed from subsequent eligible purchases within the reporting window.
# Where two orders share a date, order_id provides a deterministic ordering, not event chronology.
order_sequence = valid_orders.sort_values(["order_date", "order_id"]).copy()
order_sequence["Prior_observed_orders"] = order_sequence.groupby("customer_id").cumcount()
order_sequence["Purchase_type"] = order_sequence["Prior_observed_orders"].eq(0).map(
    {True: "First observed", False: "Subsequent observed"})
purchase_mix = (order_sequence.groupby(["channel_clean", "Purchase_type"])["order_id"]
                .count().unstack(fill_value=0).reindex(channels))
purchase_mix["Total eligible orders"] = purchase_mix.sum(axis=1)
purchase_mix["Subsequent share"] = (
    purchase_mix["Subsequent observed"] / purchase_mix["Total eligible orders"])

# Cancellation/return shares use unique, non-disputed orders, including all recorded statuses.
status_counts = (orders_clean.groupby(["channel_clean", "status"])["order_id"]
                 .nunique().unstack(fill_value=0).reindex(channels))
status_counts["Noncompleted orders"] = (
    status_counts.get("cancelled", 0) + status_counts.get("returned", 0))
status_counts["Unique orders"] = orders_clean.groupby("channel_clean")["order_id"].nunique().reindex(channels)
status_counts["Cancelled/returned share"] = (
    status_counts["Noncompleted orders"] / status_counts["Unique orders"])

channel_quality = purchase_mix[["First observed", "Subsequent observed", "Subsequent share"]].join(
    status_counts[["Noncompleted orders", "Unique orders", "Cancelled/returned share"]])
print("Last-click purchase mix and recorded order status (distinct denominators):")
display(channel_quality.round(4))
print("First observed means first eligible purchase within Jan–Jun, not first lifetime purchase. "
      "Order-status shares do not measure recoverable lost revenue.")

## 9. Question 2 — measurable session funnel

Session-to-cart is the share of observed sessions containing an add-to-cart action. Checkout-start / add-to-cart is an **aggregate count ratio**, not a verified progression of the same sessions: the extract provides no session IDs. Missing dates remain unobserved and no checkout-to-purchase conversion is inferred.

In [ ]:
# Measure observed session-funnel stages only; checkout starts are not linked to completed orders.
funnel = sessions.groupby("channel_clean")[["sessions", "add_to_cart", "checkouts"]].sum().reindex(channels)
funnel["Session → cart"] = funnel["add_to_cart"] / funnel["sessions"]
funnel["Checkout starts / carts (aggregate)"] = funnel["checkouts"] / funnel["add_to_cart"]
funnel["Session → checkout"] = funnel["checkouts"] / funnel["sessions"]
funnel["Lost before cart"] = funnel["sessions"] - funnel["add_to_cart"]
funnel["Cart–checkout count gap"] = funnel["add_to_cart"] - funnel["checkouts"]

expected = pd.date_range("2026-01-01", "2026-06-30")
missing_session_dates = expected.difference(pd.DatetimeIndex(sessions["date"].unique()))
print("Session dates missing:", [d.strftime("%Y-%m-%d") for d in missing_session_dates])

session_total = int(funnel["sessions"].sum())
cart_total = int(funnel["add_to_cart"].sum())
checkout_total = int(funnel["checkouts"].sum())
print(f"Session → cart: {cart_total:,}/{session_total:,} = {cart_total/session_total:.2%}")
print(f"Aggregate checkout-start / add-to-cart ratio: {checkout_total:,}/{cart_total:,} = {checkout_total/cart_total:.2%}")
print("Largest measured loss is before add-to-cart; causes cannot be inferred from counts alone.")
display(funnel.round(4))


# Benchmark Q2 Meta's recorded add-to-cart rate against Google Ads at the same traffic volume.
# This is a deterministic rate-gap scenario; it is not incremental sessions, orders, or sales.
q2_observed_sessions = sessions.loc[sessions["date"].between("2026-04-01", "2026-06-30")]
q2_stage_counts = (q2_observed_sessions.groupby("channel_clean")
    [["sessions", "add_to_cart"]].sum())
meta_q2_sessions = int(q2_stage_counts.loc["Meta", "sessions"])
meta_cart_rate = q2_stage_counts.loc["Meta", "add_to_cart"] / meta_q2_sessions
google_cart_rate = (q2_stage_counts.loc["Google Ads", "add_to_cart"] /
                    q2_stage_counts.loc["Google Ads", "sessions"])
cart_gap_scenario = meta_q2_sessions * (google_cart_rate - meta_cart_rate)
print(f"Q2 benchmark (not forecast): {meta_q2_sessions:,} Meta sessions × "
      f"({google_cart_rate:.2%} Google Ads − {meta_cart_rate:.2%} Meta) "
      f"= {cart_gap_scenario:,.0f} additional add-to-cart sessions, arithmetically.")


## 10. Monthly adjusted ROAS and campaign delivery

Calculate monthly channel revenue using signed financial records (positive completed sales and all 12 negative adjustments), while purchase counts remain based on positive completed orders only. Monthly median ROAS describes variation, not marginal returns. Campaign-level CTR, CPC and CPM are available; campaign-level revenue attribution is not.

In [ ]:
# Monthly channel economics use signed adjusted revenue; campaign metrics remain delivery-only.
revenue_records["Month"] = revenue_records["order_date"].dt.to_period("M").astype(str)
spend_clean["Month"] = spend_clean["date"].dt.to_period("M").astype(str)
monthly = pd.concat([
    revenue_records.groupby(["Month", "channel_clean"])["revenue"].sum().rename("Revenue (€)"),
    spend_clean.groupby(["Month", "channel_clean"])["spend"].sum().rename("Spend (€)")
], axis=1).fillna(0).reset_index()
monthly["ROAS"] = monthly["Revenue (€)"] / monthly["Spend (€)"].replace(0, float("nan"))
print("Monthly ROAS across paid channels:")
display(monthly.pivot(index="Month", columns="channel_clean", values="ROAS")
        [["Email", "Affiliate", "Google Ads", "Meta"]].round(2))


# Median monthly ROAS is not the same metric as total revenue / total spend.
monthly_paid = monthly.loc[monthly["channel_clean"].isin(["Email", "Affiliate", "Google Ads", "Meta"])]
roas_stability = (monthly_paid.groupby("channel_clean")["ROAS"]
    .agg(Median_monthly_ROAS="median", Min_monthly_ROAS="min",
         Max_monthly_ROAS="max", Months_observed="count")
    .reindex(["Email", "Affiliate", "Google Ads", "Meta"]))
monthly_compare = monthly.pivot(index="Month", columns="channel_clean", values="ROAS")
affiliate_beats_meta = int((monthly_compare["Affiliate"] > monthly_compare["Meta"]).sum())
print(f"Affiliate outperformed Meta on reported monthly ROAS in {affiliate_beats_meta}/6 months.")
display(roas_stability.round(3))

campaigns = (spend_clean.groupby(["channel_clean", "campaign"], as_index=False)
             [["spend", "clicks", "impressions"]].sum())
campaigns["CTR"] = campaigns["clicks"] / campaigns["impressions"].replace(0, float("nan"))
campaigns["CPC (€)"] = campaigns["spend"] / campaigns["clicks"].replace(0, float("nan"))
campaigns["CPM (€)"] = campaigns["spend"] * 1000 / campaigns["impressions"].replace(0, float("nan"))
print("Campaign traffic and cost efficiency ONLY — no order/campaign mapping available:")
display(campaigns.round(3))

paid_monthly = (monthly.loc[monthly["channel_clean"] != "Organic"]
    .groupby("Month")[["Revenue (€)", "Spend (€)"]].sum())
paid_monthly.plot(figsize=(8, 3), marker="o", title="Paid-channel attributed revenue vs ad spend")
plt.ylabel("€")
plt.tight_layout()
plt.show()

## 11. Customer purchasing patterns and Email's observed role

Buyer frequency is based on eligible orders, while acquisition-channel cohorts use the customer table. Six-month repeat rates have unequal exposure periods and must not be interpreted as lifetime retention. A 30-day follow-up metric is reported only for buyers whose first observed order permits a full window. Email's subsequent-order share is descriptive, not an estimate of Email's incremental effect.

In [ ]:
# Attribute cohort summaries to customer acquisition channel, not each order's last-click channel.
# Restrict 30-day repeats to buyers with an observable full follow-up window.
# Interpret Email subsequent orders as purchase-history context, not incremental campaign effect.
customer_orders = (valid_orders.sort_values(["order_date", "order_id"])
    .groupby("customer_id").agg(
        Orders=("order_id", "size"), Revenue_EUR=("revenue", "sum"),
        First_order=("order_date", "min"), Last_order=("order_date", "max")))
customer_orders["Repeat_buyer"] = customer_orders["Orders"].gt(1)

customer_info = (customers[["customer_id", "registration_date", "channel_clean"]]
    .drop_duplicates("customer_id").set_index("customer_id")
    .rename(columns={"channel_clean": "Acquisition_channel"}))
customer_orders = customer_orders.join(customer_info, how="left")
unmatched_buyers = int(customer_orders["Acquisition_channel"].isna().sum())
print("Observed buyers:", len(customer_orders),
      "| repeat buyers:", int(customer_orders["Repeat_buyer"].sum()),
      "| unmatched acquisition records:", unmatched_buyers)

matched = customer_orders.dropna(subset=["Acquisition_channel"])
customer_cohorts = (matched.groupby("Acquisition_channel")
    .agg(Buyers=("Orders", "size"), Repeat_buyers=("Repeat_buyer", "sum"),
         Completed_orders=("Orders", "sum"), Revenue_EUR=("Revenue_EUR", "sum"))
    .reindex(channels))
customer_cohorts["Observed_repeat_rate"] = customer_cohorts["Repeat_buyers"] / customer_cohorts["Buyers"]
customer_cohorts["Revenue_per_buyer_EUR"] = customer_cohorts["Revenue_EUR"] / customer_cohorts["Buyers"]

first_cutoff = pd.Timestamp("2026-06-30") - pd.Timedelta(days=30)
eligible_30d = customer_orders["First_order"].le(first_cutoff)
first_dates = valid_orders["customer_id"].map(customer_orders["First_order"])
second_in_30d = (valid_orders["order_date"].gt(first_dates) &
                 valid_orders["order_date"].le(first_dates + pd.Timedelta(days=30)))
first_order_ids = (valid_orders.sort_values(["order_date", "order_id"])
                   .drop_duplicates("customer_id")["order_id"])
subsequent_30d = (valid_orders["order_date"].le(first_dates + pd.Timedelta(days=30)) &
                  ~valid_orders["order_id"].isin(first_order_ids))
repeated_30d_ids = set(valid_orders.loc[subsequent_30d, "customer_id"])
customer_orders["Followup_30d_available"] = eligible_30d
customer_orders["Repeat_within_30d"] = customer_orders.index.isin(repeated_30d_ids) & eligible_30d
customer_cohorts["Buyers_with_30d_followup"] = (customer_orders.loc[
    customer_orders["Followup_30d_available"] & customer_orders["Acquisition_channel"].notna()]
    .groupby("Acquisition_channel").size().reindex(channels, fill_value=0))
customer_cohorts["30d_repeat_buyers"] = (customer_orders.loc[
    customer_orders["Repeat_within_30d"] & customer_orders["Acquisition_channel"].notna()]
    .groupby("Acquisition_channel").size().reindex(channels, fill_value=0))
customer_cohorts["30d_repeat_rate"] = (
    customer_cohorts["30d_repeat_buyers"] /
    customer_cohorts["Buyers_with_30d_followup"].replace(0, float("nan")))
print("Acquisition-channel cohorts: observed repeat rate vs fixed 30-day follow-up:")
display(customer_cohorts.round(3))

ordered = valid_orders.sort_values(["order_date", "order_id"]).copy()
ordered["Previous_eligible_orders"] = ordered.groupby("customer_id").cumcount()
email_orders = ordered.loc[ordered["channel_clean"].eq("Email")]
email_repeat_n = int(email_orders["Previous_eligible_orders"].gt(0).sum())
print(f"Email orders after an earlier observed eligible purchase: {email_repeat_n} "
      f"/ {len(email_orders)} = {email_repeat_n / len(email_orders):.1%}")

focus_id = orders_clean.loc[orders_clean["order_id"].eq("O502653"), "customer_id"].iloc[0]
focus_history = orders_clean.loc[orders_clean["customer_id"].eq(focus_id),
    ["order_date", "order_id", "channel_clean", "revenue", "status"]].sort_values("order_date")
print("Large Meta transaction customer:", focus_id)
display(focus_history)
print("Interpretation: Six-month observed spend is not LTV; last-click attribution is not causation.")


## 12. Meta revenue concentration — adjusted-revenue sensitivity

Keep the €24,990 completed purchase in the primary adjusted revenue. Remove it **only from a diagnostic comparison**. Both cases retain the same marketing-spend denominator and the negative adjustments included in the final report.

In [ ]:
# Concentration diagnostic: same adjusted-revenue policy in both cases.
meta_spend = float(channel_report.loc["Meta", "Spend (€)"])
meta_with = float(channel_report.loc["Meta", "Revenue (€)"])
meta_order_amount = float(valid_orders.loc[valid_orders["order_id"] == meta_id, "revenue"].iloc[0])
meta_without = meta_with - meta_order_amount
scenario_comparison = pd.DataFrame({
    "Meta adjusted revenue (€)": [meta_with, meta_without],
    "Meta attributed ROAS": [meta_with/meta_spend, meta_without/meta_spend]
}, index=["Primary: include valid sale", "Sensitivity only: exclude sale"])
display(scenario_comparison.round(3))
assert abs(meta_with-62646.30)<0.01
assert abs(meta_with/meta_spend-1.3718635518451614)<1e-6
assert abs(meta_without/meta_spend-0.8246186202113606)<1e-6

## 13. Question 3 — financially constrained Q3 budget choice

The management workbook recommends **one conditional option (B: Affiliate-only)**, with two alternatives for comparison. Increase Affiliate's **own Q2 budget by 10% (€624.09)**, funded by Meta; this is *not* a 10% Meta-spend reduction. All three alternatives conserve the €52,853.72 quarterly paid budget.

- **A — 60/40 diversification:** €374.45 Affiliate + €249.64 Google, funded by Meta.
- **B — Affiliate only:** €624.09 to Affiliate, the preferred conditional pilot.
- **C — Google only:** €624.09 to Google, another unapproved alternative.

**Approval:** `HOLD UPLIFT` until Finance and Marketing approve a contribution margin, reporting/returns lag, return/cancellation guardrail, and the spending cap. A 42-day (six-week) review is not a powered experiment. ROAS-implied break-even margins are accounting ratios, **not measured incremental profitability**.

In [ ]:
# Reconcile Q2 adjusted revenue and spending, then test equal-envelope allocation alternatives.
q2_start, q2_end = "2026-04-01", "2026-06-30"
q2_spend = (spend_clean.loc[spend_clean["date"].between(q2_start, q2_end)]
            .groupby("channel_clean")["spend"].sum().reindex(channels, fill_value=0))
q2_revenue = (revenue_records.loc[revenue_records["order_date"].between(q2_start, q2_end)]
              .groupby("channel_clean")["revenue"].sum().reindex(channels, fill_value=0))
q2_purchases = (valid_orders.loc[valid_orders["order_date"].between(q2_start, q2_end)]
                .groupby("channel_clean")["order_id"].count().reindex(channels, fill_value=0))
q2_economics = pd.DataFrame({"Q2 spend (€)":q2_spend,"Q2 adjusted sales (€)":q2_revenue,
                             "Positive purchases":q2_purchases})
q2_economics["Attributed ROAS"] = q2_revenue / q2_spend.replace(0, float("nan"))
q2_economics["ROAS-implied break-even margin"] = q2_spend / q2_revenue.replace(0, float("nan"))
print("Quarterly adjusted attribution and historical ROAS (not profit):")
display(q2_economics.round(3))

# 10% of the RECIPIENT Affiliate's actual Q2 budget; this transfer is taken from Meta.
affiliate_uplift_fraction = 0.10
moved = round(float(q2_spend["Affiliate"]) * affiliate_uplift_fraction, 2)
q2_total = round(float(q2_spend.sum()),2)
assert abs(moved-624.09)<0.01
assert abs(q2_total-52853.72)<0.01

# Each scenario is mutually exclusive, not a staged sequence.
allocation_shares = {
    "Q2 unchanged reference": None,
    "A | 60/40 Affiliate/Google": (0.60, 0.40),
    "B | Affiliate only — conditional preference": (1.00, 0.00),
    "C | Google Ads only": (0.00, 1.00)
}
scenario_rows=[]
for option, weights in allocation_shares.items():
    candidate=q2_spend.copy()
    if weights is not None:
        to_affiliate=round(moved*weights[0],2)
        to_google=round(moved-to_affiliate,2)  # residual avoids rounding drift
        candidate.loc["Meta"] -= moved
        candidate.loc["Affiliate"] += to_affiliate
        candidate.loc["Google Ads"] += to_google
    assert abs(candidate.sum()-q2_total)<0.005, "Quarterly envelope was not preserved"
    assert candidate.ge(0).all(), "Negative allocation"
    scenario_rows.append({"Scenario":option, "Meta (€)":candidate["Meta"],
                          "Affiliate (€)":candidate["Affiliate"],
                          "Google Ads (€)":candidate["Google Ads"],
                          "Email (€)":candidate["Email"],
                          "Total paid (€)":candidate.sum()})
allocation_options=pd.DataFrame(scenario_rows).set_index("Scenario")
print("Q2-equivalent allocations; no future revenue/profit forecast:")
display(allocation_options.round(2))

# B remains the existing recommendation. Neither the quarter-equivalent scenario nor the
# six-week cap is approved spending. Finance/Marketing sign-off and approved economics are missing.
budget = pd.DataFrame({"Q2 actual (€)":q2_spend,
                       "B: conditional quarterly amount (€)":q2_spend.copy()})
budget.loc["Meta", "B: conditional quarterly amount (€)"] -= moved
budget.loc["Affiliate", "B: conditional quarterly amount (€)"] += moved
budget["B transfer (€)"] = budget["B: conditional quarterly amount (€)"]-budget["Q2 actual (€)"]
budget["Q2 attributed ROAS"] = q2_economics["Attributed ROAS"]
budget["ROAS-implied break-even margin"] = q2_economics["ROAS-implied break-even margin"]

pilot_days, q2_days = 42, 91
six_week = pd.DataFrame({"Q2-paced 42-day reference cap (€)":
    q2_spend.map(lambda amount: round(float(amount)*pilot_days/q2_days,2))})
pilot_uplift = round(float(six_week.loc["Affiliate", "Q2-paced 42-day reference cap (€)"])*affiliate_uplift_fraction,2)
six_week["B conditional cap (€)"] = six_week["Q2-paced 42-day reference cap (€)"].copy()
six_week.loc["Meta", "B conditional cap (€)"] -= pilot_uplift
six_week.loc["Affiliate", "B conditional cap (€)"] += pilot_uplift

approval_inputs = {"Approved contribution margin": None,
                   "Approved reporting/return lag (days)": None,
                   "Approved max return/cancel rate": None,
                   "Finance and Marketing sign-off": "Pending"}
approval_status = "HOLD UPLIFT"
print("Six-week exposure caps (assuming Q2 daily pacing):")
display(six_week.round(2))
print("Approval controls:",approval_inputs,"| Status:",approval_status)
print("Option B is a capped *proposal*, not approved incremental investment or causal evidence.")

assert abs(float(q2_revenue.drop("Organic").sum())-103694.86)<0.01
assert abs(float(q2_economics.loc["Affiliate","ROAS-implied break-even margin"])-0.6153252307865229)<1e-6
assert abs(float(q2_economics.loc["Google Ads","ROAS-implied break-even margin"])-0.8271957201538146)<1e-6
assert abs(pilot_uplift-288.04)<0.01
assert len(allocation_options)==4
assert approval_status=="HOLD UPLIFT"

## 14. Source-row exception and adjustment register

Keep a row-level trail for exclusions **and** included financial adjustments. Disputed `O500891` and exact duplicate copies are quarantined. Positive cancelled/returned orders and zero-value rows are excluded. Negative-value entries are **included in adjusted revenue** but flagged for accounting verification; they are not counted as purchases. The €24,990 completed Meta order remains included in primary calculations and does not enter the exclusion register.

In [ ]:
# Record *treatment*, rather than inaccurately listing recognised negative adjustments as excluded.
positive_noncompleted = orders_clean.loc[
    orders_clean["status"].ne("completed") & orders_clean["revenue"].ge(0)].copy()
positive_noncompleted["Treatment"] = "EXCLUDE — positive/zero cancelled or returned"
zero_completed = orders_clean.loc[
    orders_clean["status"].eq("completed") & orders_clean["revenue"].eq(0)].copy()
zero_completed["Treatment"] = "EXCLUDE — zero-value completed"
negative_audit = negative_adjustments.copy()
negative_audit["Treatment"] = "INCLUDE signed value in adjusted revenue; not a positive purchase"
conflicting_audit = conflicting_order.copy()
conflicting_audit["Treatment"] = "EXCLUDE — contradictory O500891 values"
identical_audit = extra_order_copies.copy()
identical_audit["Treatment"] = "EXCLUDE — repeated order ID"
parts=[positive_noncompleted, zero_completed, negative_audit, conflicting_audit, identical_audit]
exception_register = pd.concat([x for x in parts if not x.empty],ignore_index=True)
print("Source exception register (includes retained negative adjustments):")
display(exception_register["Treatment"].value_counts().rename_axis("Treatment").to_frame("Rows"))
print("Signed adjustment count and value:",len(negative_audit),
      f"€{negative_audit['revenue'].sum():,.2f}")
print("Exact duplicate marketing spend rows provisionally removed:",len(extra_spend_copies),
      f"€{extra_spend_copies['spend'].sum():,.2f}")

financial_sensitivities = pd.DataFrame([
    ["O500891 conflicting values", "€101.21 / €18,450", "Both excluded pending finance verification"],
    ["All recorded negative adjustments", f"€{negative_adjustments['revenue'].sum():,.2f}",
      "Deducted from financial revenue; excluded from purchase counts"],
    ["Identical spend rows", f"€{extra_spend_copies['spend'].sum():,.2f}",
      "Provisional deduplication; ledger confirmation required"],
    ["O502653 completed Meta order", f"€{meta_order_amount:,.2f}",
      "Included in primary; removed only in diagnostic sensitivity"],
    ["Missing session dates", ", ".join(d.strftime("%d %b") for d in missing_session_dates),
      "Unobserved sessions, not zero traffic"],
],columns=["Issue","Measured scale","Treatment"])
display(financial_sensitivities)

## 15. Decision limits and verification priorities

- **Attribution:** Last-click order channel is not campaign attribution, and neither establishes incremental uplift. Campaign spend supports delivery measures (CTR, CPC, CPM), not campaign ROAS.
- **Revenue and quality:** The primary totals deduct 12 signed adjustments (−€719.51), including negative cancelled/returned entries. Positive cancelled/returned orders remain excluded. These provisional financial policies need settlement-ledger confirmation, especially conflicting `O500891` (€101.21 versus €18,450).
- **Large completed transaction:** Keep `O502653` (€24,990 Meta) in primary sales; use the single-order exclusion solely to quantify concentration risk.
- **Spend source:** Twenty exact matching spend rows (−€1,268.45 provisional deduplication) require transaction-level confirmation.
- **Funnel:** No session-to-order key is available; 11–13 April session observations are missing, not zero. Checkout-to-purchase and verified within-session abandonment are not identifiable.
- **Customers:** 15 positive-purchase buyers lack acquisition records; six-month observed repeats are not lifetime retention or LTV.
- **Contribution margin:** No verified COGS, fulfilment, return-lag or contribution-margin inputs support incremental profit claims. Historical attributed ROAS ratios provide accounting break-even indicators only.

**Recommendation:** Prioritise a Google Ads profitability review, maintain Email lifecycle efforts, and compare three constant-budget Meta transfer options. Prefer **B (Affiliate-only)** only conditionally. The release decision remains **HOLD UPLIFT** until Finance/Marketing sign off on margin, reporting lag, return/cancellation rules and spend caps. The six-week pilot is an observational review, not a causal experiment.

## 16. Reconciliation against the final management report

Checks below compare independently calculated notebook totals against the **published report's stated targets**; they do not validate source-ledger integrity or causal conclusions. A mismatch must be investigated, not rounded away.

In [ ]:
# Central, independently recomputed checks against the final Excel benchmark and internal identities.
channel_revenue_by_month = monthly["Revenue (€)"].sum()
channel_spend_by_month = monthly["Spend (€)"].sum()
paid_revenue = float(channel_report.loc[channels[:-1], "Revenue (€)"].sum())
paid_spend = float(channel_report.loc[channels[:-1], "Spend (€)"].sum())
checks = {
    "Unique orders; disputed O500891 quarantined":
        orders_clean["order_id"].is_unique and "O500891" not in orders_clean["order_id"].values,
    "Provisional identical spend rows removed":
        not spend_clean.duplicated(subset=spend_fields).any() and len(extra_spend_copies)==20,
    "Completed positive purchases and all 12 signed adjustments separated":
        valid_orders["status"].eq("completed").all() and valid_orders["revenue"].gt(0).all()
        and len(valid_orders)==3065 and len(negative_adjustments)==12,
    "Positive sales minus adjustments equals adjusted total":
        abs(valid_orders["revenue"].sum()-212282.52)<0.01
        and abs(negative_adjustments["revenue"].sum()+719.51)<0.01
        and abs(revenue_records["revenue"].sum()-211563.01)<0.01,
    "Five channel revenue totals agree with management report":
        all(abs(float(channel_report.loc[c,"Revenue (€)"])-target)<0.01 for c,target in {
            "Email":34210.03, "Affiliate":18913.69, "Google Ads":52495.83,
            "Meta":62646.30, "Organic":43297.16}.items()),
    "Paid revenue/spend and ROAS reconcile":
        abs(paid_revenue-168265.85)<0.01 and abs(paid_spend-103861.75)<0.01
        and abs(paid_revenue/paid_spend-1.6200945006222214)<1e-6,
    "Monthly signed revenue and spend reconcile":
        abs(channel_revenue_by_month-revenue_records["revenue"].sum())<0.01
        and abs(channel_spend_by_month-spend_clean["spend"].sum())<0.01,
    "Meta sensitivity retains €24,990 in primary":
        abs(meta_with-62646.30)<0.01 and abs(meta_without-37656.30)<0.01,
    "Q2 financial baseline reconciles":
        abs(q2_total-52853.72)<0.01
        and abs(float(q2_revenue.drop("Organic").sum())-103694.86)<0.01,
    "Q2 transfer, all three alternatives and B recommendation reconcile":
        abs(moved-624.09)<0.01 and all(abs(allocation_options["Total paid (€)"]-q2_total)<0.005)
        and abs(float(budget.loc["Affiliate","B: conditional quarterly amount (€)"])-6865.02)<0.01
        and abs(float(budget.loc["Meta","B: conditional quarterly amount (€)"])-23101.69)<0.01,
    "Six-week pilot cap remains unapproved":
        abs(pilot_uplift-288.04)<0.01 and approval_status=="HOLD UPLIFT",
    "Customer cohort coverage reconciles":
        len(customer_orders)==2020 and unmatched_buyers==15
        and int(customer_cohorts["Buyers"].sum())==2005,
    "Funnel stage counts reconcile":
        int(funnel["sessions"].sum())==293692
        and int(funnel["add_to_cart"].sum())==16441
        and int(funnel["checkouts"].sum())==6490,
}
check_results = pd.DataFrame({"Control":list(checks),
    "Result":["PASS" if x else "FAIL" for x in checks.values()]})
display(check_results)
assert all(checks.values()), "Reconciliation FAILED: examine the inputs and exception policy."
print(f"Passed: {sum(checks.values())} / {len(checks)} source-driven checks")

## Publication and reproducibility note

This notebook's code is aligned with `marketing_management_report.xlsx` on signed revenue (12 adjustments), channel/monthly attributed ROAS, Meta sensitivity, budget transfers, three alternatives, the six-week cap, and financial approval status.

**Execution limitation:** `Shop_online_data.xlsx` is not included and was not available when this revised notebook was prepared. Therefore the revised notebook's prior outputs were cleared rather than retaining misleading results from a different accounting policy. The Excel report contains the actual verified reference values. With authorised access to the synthetic source file, run the code and inspect the final PASS/FAIL reconciliation table.

Historical attributed performance does not establish campaign-level causal returns, marginal ROAS or contribution profit.